# Heart Disease — Pipeline ejecutable (Tarea 1.2 / 2.x)

Notebook ejecutable en Google Colab que reproduce exactamente el pipeline de `main.py`, paso por paso, con explicaciones de por qué cada paso está en ese orden.

**Para abrir en Colab sin este repo ya clonado**, corre esta celda primero (quita el `#`):
```
# !git clone https://github.com/josuedcpy-boop/MAI540.git
# %cd MAI540
```

Referencias: `README_TECNICO.md` (orden del pipeline y por qué evita fuga), `Contexto.md` (reglas y restricciones vigentes), `BITACORA.md` (evidencia numérica de cada decisión).

In [1]:
import sys
from pathlib import Path
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

## Paso 1: Cargar los datos

Ruta robusta: funciona tanto localmente (ejecutando desde la raíz del repo) como en Colab después de clonar.

In [2]:
CANDIDATOS = [Path("data/datos.csv"), Path("MAI540/data/datos.csv")]
DATA = next((p for p in CANDIDATOS if p.exists()), None)
if DATA is None:
    raise FileNotFoundError(
        "No se encontro data/datos.csv. En Colab, corre primero:\n"
        "!git clone https://github.com/josuedcpy-boop/MAI540.git\n"
        "%cd MAI540"
    )

df = pd.read_csv(DATA)
print(f"Filas cargadas: {len(df):,}")
df.head()

Filas cargadas: 7,000


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,58.0,1.0,4.0,128.0,216.0,0.0,2.0,131.0,1.0,2.2,2.0,3.0,7.0,1
1,45.0,1.0,2.0,128.0,307.0,0.0,2.0,171.0,0.0,0.0,1.0,0.0,3.0,0
2,59.0,1.0,4.0,138.0,271.0,0.0,2.0,182.0,0.0,0.0,1.0,0.0,3.0,0
3,44.0,1.0,3.0,120.0,226.0,0.0,0.0,169.0,0.0,0.0,1.0,0.0,3.0,0
4,54.0,1.0,4.0,109.0,237.0,0.0,0.0,125.0,1.0,2.8,2.0,1.0,7.0,1


## Paso 2: Eliminar duplicados — ANTES de dividir train/test

`datos.csv` tiene filas duplicadas exactas (copias del mismo paciente). Si se dividiera primero y se
deduplicara después, copias del mismo paciente podrían quedar repartidas entre train y test — el
modelo "memorizaría" en entrenamiento una fila casi idéntica a la que luego se evalúa en test. Por
eso se deduplica **antes** de cualquier split.

In [3]:
filas_antes_dedup = len(df)
df = df.drop_duplicates().reset_index(drop=True)
filas_despues_dedup = len(df)
filas_duplicadas_eliminadas = filas_antes_dedup - filas_despues_dedup

print(f"Filas antes de quitar duplicados: {filas_antes_dedup:,}")
print(f"Filas duplicadas eliminadas: {filas_duplicadas_eliminadas:,}")
print(f"Filas despues de quitar duplicados: {filas_despues_dedup:,}")

Filas antes de quitar duplicados: 7,000
Filas duplicadas eliminadas: 1,669
Filas despues de quitar duplicados: 5,331


## Paso 3: Diagnóstico de valores faltantes (solo lectura)

Se cuentan antes de que el pipeline los impute, para que quede visible qué se está "arreglando".

In [4]:
missing_counts = df.isna().sum()
missing_counts = missing_counts[missing_counts > 0]
print("Valores faltantes por columna:")
print(missing_counts)

Valores faltantes por columna:
ca      65
thal    37
dtype: int64


## Paso 4: Indicador de que `thal` era faltante

Se calcula fila por fila a partir del propio valor de `thal` — no usa información de otras filas ni
del target, así que es seguro calcularlo antes del split (el resultado sería idéntico si se hiciera
después).

In [5]:
df["thal_missing"] = df["thal"].isna().astype(int)
df[["thal", "thal_missing"]].head()

,thal,thal_missing
0,7.0,0
1,3.0,0
2,3.0,0
3,3.0,0
4,7.0,0


## Paso 5: Listas de columnas y chequeo de columna prohibida

`target` nunca debe usarse como predictor (fuga directa del objetivo) — es una regla absoluta,
documentada en `Contexto.md`. `X` se construye explícitamente sin incluirla.

In [6]:
numeric_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical_features = ["sex", "cp", "restecg", "exang", "thal", "fbs", "slope", "ca"]
indicator_features = ["thal_missing"]
baseline_features = ["age", "trestbps", "chol", "thalach"]

FORBIDDEN_FEATURES = {"target"}
used_features = set(numeric_features) | set(categorical_features) | set(baseline_features)
leaked = used_features & FORBIDDEN_FEATURES

if leaked:
    print(f"ADVERTENCIA - columnas prohibidas usadas: {sorted(leaked)}")
else:
    print(f"Verificacion de fuga de datos: OK (columnas prohibidas no usadas: {sorted(FORBIDDEN_FEATURES)})")

X = df[numeric_features + categorical_features + indicator_features]
y = df["target"].astype(int)

Verificacion de fuga de datos: OK (columnas prohibidas no usadas: ['target'])


## Paso 6: `train_test_split` — el punto de corte

A partir de aquí, todo lo que aprenda algún parámetro de los datos (medianas, límites de atípicos,
escalas, categorías, correlaciones para selección de variables) debe calcularse **solo** con
`X_train`/`y_train`. `random_state=42` fijo hace el split reproducible; `stratify=y` mantiene la
proporción de clases en ambos conjuntos.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,} filas | Test: {len(X_test):,} filas")

Train: 3,998 filas | Test: 1,333 filas


## Paso 7: Selección de características — correlación calculada SOLO en train

Si esto se calculara sobre el dataset completo (incluyendo test), estaríamos decidiendo qué
variables usar basándonos parcialmente en el conjunto que se supone que el modelo nunca ha visto.

In [8]:
train_con_target = X_train.copy()
train_con_target["target"] = y_train
correlacion_target = (
    train_con_target.corr(numeric_only=True)["target"].drop("target").sort_values(key=abs, ascending=False)
)
UMBRAL_CORRELACION = 0.15
variables_conservadas = correlacion_target[correlacion_target.abs() >= UMBRAL_CORRELACION].index.tolist()
variables_descartadas = correlacion_target[correlacion_target.abs() < UMBRAL_CORRELACION].index.tolist()
selected_numeric_features = [f for f in numeric_features if f in variables_conservadas]
selected_categorical_features = [f for f in categorical_features if f in variables_conservadas]

print(correlacion_target.round(4).to_string())
print(f"\nConservadas ({len(variables_conservadas)}): {variables_conservadas}")
print(f"Descartadas ({len(variables_descartadas)}): {variables_descartadas}")

thal            0.5427
exang           0.4465
ca              0.4420
thalach        -0.4313
cp              0.4244
oldpeak         0.4179
slope           0.3299
sex             0.3039
age             0.2163
restecg         0.1580
trestbps        0.1231
chol            0.0769
fbs             0.0335
thal_missing    0.0229

Conservadas (10): ['thal', 'exang', 'ca', 'thalach', 'cp', 'oldpeak', 'slope', 'sex', 'age', 'restecg']
Descartadas (4): ['trestbps', 'chol', 'fbs', 'thal_missing']


## Transformador para atípicos: `RecorteIQR`

`RobustScaler` por sí solo no "trata" los atípicos, solo evita que dominen la escala. Este
transformador los recorta (winsoriza) a los límites del IQR **calculados solo en `fit()`** (train),
antes de escalar. Evidencia de que esto ayuda (no baja accuracy, sube recall): `BITACORA.md`
sección 11.

In [9]:
class RecorteIQR(BaseEstimator, TransformerMixin):
    """Recorta cada columna a [Q1 - k*IQR, Q3 + k*IQR]. Limites calculados solo en fit() (train)."""

    def __init__(self, k=1.5):
        self.k = k

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        q1, q3 = X.quantile(0.25), X.quantile(0.75)
        iqr = q3 - q1
        self.lower_ = (q1 - self.k * iqr).to_numpy()
        self.upper_ = (q3 + self.k * iqr).to_numpy()
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for i, columna in enumerate(X.columns):
            X[columna] = X[columna].clip(self.lower_[i], self.upper_[i])
        return X.to_numpy()

## Función auxiliar para reportar métricas

In [10]:
def print_report(titulo, n_variables, y_true, y_pred):
    print(f"=== {titulo} ===")
    print(f"Variables usadas: {n_variables}")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1:        {f1_score(y_true, y_pred):.4f}")
    print("Matriz de confusion:")
    print(confusion_matrix(y_true, y_pred))
    print()

## Modelo ANTES — punto de partida (fijo, no se modifica)

Solo 4 variables numéricas, sin escalar, sin balanceo de clases. Es la evidencia histórica de la
Tarea 1.2, congelada — el pipeline la sigue calculando igual para poder comparar.

In [11]:
baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
baseline_model.fit(X_train[baseline_features], y_train)
baseline_pred = baseline_model.predict(X_test[baseline_features])
print_report("ANTES (punto de partida)", len(baseline_features), y_test, baseline_pred)

=== ANTES (punto de partida) ===
Variables usadas: 4
Accuracy:  0.7097
Precision: 0.7164
Recall:    0.6620
F1:        0.6882
Matriz de confusion:
[[519 169]
 [218 427]]



## Modelo DESPUÉS — 14 variables, preprocesamiento diferenciado, balanceo de clases

`ColumnTransformer` con tres ramas: numérica (imputar → recortar atípicos → escalar), categórica
(imputar → one-hot), e indicador (passthrough). Todo ajustado únicamente con `X_train`.

In [12]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("recorte", RecorteIQR(k=1.5)),
        ("scaler", RobustScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
    ("ind", "passthrough", indicator_features),
])
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")),
])
final_model.fit(X_train, y_train)
final_pred = final_model.predict(X_test)
print_report("DESPUES (mejora equilibrada)", len(numeric_features) + len(categorical_features) + len(indicator_features), y_test, final_pred)

=== DESPUES (mejora equilibrada) ===
Variables usadas: 14
Accuracy:  0.8740
Precision: 0.8804
Recall:    0.8558
F1:        0.8679
Matriz de confusion:
[[613  75]
 [ 93 552]]



## Veredicto — Recall como criterio principal (no accuracy)

Un falso negativo (no detectar una condición real) es más grave que una falsa alarma.

In [13]:
recall_antes = recall_score(y_test, baseline_pred)
recall_despues = recall_score(y_test, final_pred)
print(f"Recall ANTES:   {recall_antes:.4f}")
print(f"Recall DESPUES: {recall_despues:.4f}")
if recall_despues > recall_antes:
    print(f"Mejora: recall subio {recall_despues - recall_antes:+.4f} (menos falsos negativos)")
else:
    print(f"Alerta: el recall no mejoro ({recall_despues - recall_antes:+.4f})")

Recall ANTES:   0.6620
Recall DESPUES: 0.8558
Mejora: recall subio +0.1938 (menos falsos negativos)


## Modelo SELECCIONADO — solo las variables con |r| >= 0.15

Mismo preprocesamiento que DESPUÉS, pero con el subconjunto de variables que pasó el criterio de
correlación del Paso 7.

In [14]:
selected_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("recorte", RecorteIQR(k=1.5)),
        ("scaler", RobustScaler()),
    ]), selected_numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), selected_categorical_features),
])
selected_model = Pipeline([
    ("preprocessor", selected_preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")),
])
selected_columnas = selected_numeric_features + selected_categorical_features
selected_model.fit(X_train[selected_columnas], y_train)
selected_pred = selected_model.predict(X_test[selected_columnas])
print_report("SELECCIONADO (solo variables con |r| >= 0.15)", len(selected_columnas), y_test, selected_pred)

=== SELECCIONADO (solo variables con |r| >= 0.15) ===
Variables usadas: 10


Accuracy:  0.8612
Precision: 0.8628
Recall:    0.8481
F1:        0.8554
Matriz de confusion:
[[601  87]
 [ 98 547]]



## Resumen comparativo

| Modelo | Variables | Accuracy | Recall |
|---|---|---|---|
| ANTES | 4 | ver arriba | ver arriba |
| DESPUÉS | 14 | ver arriba | ver arriba |
| SELECCIONADO | 10 | ver arriba | ver arriba |

## Salvaguardas de privacidad (no se ejecutan aquí para no interrumpir el notebook)

`main.py` incluye `safe_print`, una función que detiene el script si se intenta mostrar
públicamente las columnas `age`/`sex` (ver `Contexto.md` sección 5). Este notebook nunca imprime
filas individuales de pacientes — `print_report` solo recibe métricas agregadas — así que esa
salvaguarda no se invoca aquí, pero la regla sigue vigente para cualquier código que se agregue.

## Referencias
- Orden completo del pipeline y por qué evita fuga en cada paso: `README_TECNICO.md`.
- Evidencia numérica de cada decisión (imputación, atípicos, selección de características): `BITACORA.md`.
- Reglas de negocio y restricciones vigentes: `Contexto.md`.